## 08. Modeliranje: Eksperiment D (MC + embeddings + DL)

U ovom notebook-u treniramo MLP (multi-layer perceptron) nad Multi-Class (MC) podskupom proteina, koristeći ESM-2 embeddinge kao ulazne feature. Prati istu strukturu kao notebook 06 (SC + DL), ali prilagođenu multi-label klasifikaciji. 

Koristimo isti MC train/test split i isti MultiLabelBinarizer definisan u notebook-u u 03, čime osiguravamo fer poređenje sa klasičnim modelima iz notebook-a 07.

### Uvoz biblioteka

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import(f1_score, hamming_loss, accuracy_score, classification_report)

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Koristi se uredjaj: {DEVICE}")

Koristi se uredjaj: cpu


### Konfiguracija i putanje

In [2]:
DATA_DIR = "../data/processed"
FEATURES_DIR = "../data/features"
RESULTS_DIR = "../data/results"
MODELS_DIR = "../data/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

### Učitavanje podataka

In [3]:
# Koristimo isti MC train/test split i isti MultiLabelBinarizer definisani u notebook-u 03 
# - fer poređenje klasicnog ML i DL pristupa

# Entry liste za MC train/test skup (03 notebook)
entries_mc_train = pd.read_csv(os.path.join(FEATURES_DIR, "mc_train_entries.csv"))["Entry"].values
entries_mc_test = pd.read_csv(os.path.join(FEATURES_DIR, "mc_test_entries.csv"))["Entry"].values

# MultiLabelBinarizer fit-ovan jednom u 03
mlb = joblib.load(os.path.join(FEATURES_DIR, "mc_label_binarizer.pkl"))
label_columns = list(mlb.classes_)
NUM_CLASSES = len(label_columns)

# Labels
mc_labels_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_labels.csv"), index_col="Entry")

y_mc_train = mc_labels_df.loc[entries_mc_train, label_columns].values.astype(np.float32)
y_mc_test = mc_labels_df.loc[entries_mc_test, label_columns].values.astype(np.float32)

print(f"Train skup: {len(entries_mc_train)} proteina")
print(f"Test skup:  {len(entries_mc_test)} proteina")
print(f"Klase: {label_columns}")
print(f"y_mc_train: {y_mc_train.shape} | y_mc_test: {y_mc_test.shape}")

Train skup: 5643 proteina
Test skup:  1400 proteina
Klase: ['Hydrolase', 'Receptor', 'Structural protein', 'Transcription factor', 'Transport protein']
y_mc_train: (5643, 5) | y_mc_test: (1400, 5)


In [4]:
# ESM2 embeddinzi (izracunati u 05)
embeddings_df = pd.read_csv(os.path.join(FEATURES_DIR, "esm2_embeddings.csv"), index_col="Entry")

X_mc_train = embeddings_df.loc[entries_mc_train].values.astype(np.float32)
X_mc_test = embeddings_df.loc[entries_mc_test].values.astype(np.float32)

EMBEDDING_DIM = X_mc_train.shape[1]

print(f"Dimenzija embeddinga: {EMBEDDING_DIM}")
print(f"X_sc_train: {X_mc_train.shape} | X_sc_test: {X_mc_test.shape}")

Dimenzija embeddinga: 320
X_sc_train: (5643, 320) | X_sc_test: (1400, 320)


### Validacioni split

In [5]:
# Stratifikacija po broju labela po proteinu
label_cardinality = y_mc_train.sum(axis=1)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_mc_train, y_mc_train,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=label_cardinality
)

print(f"Trening podskup:    {X_tr.shape[0]} proteina")
print(f"Validacioni podskup: {X_val.shape[0]} proteina")

Trening podskup:    4796 proteina
Validacioni podskup: 847 proteina


### Skaliranje embeddinga

Standardizacija (fitovana isključivo na trening podskupu) ubrzava konvergenciju.

In [6]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_mc_test)

joblib.dump(scaler, os.path.join(MODELS_DIR, "mc_dl_embedding_scaler.pkl"))

['../data/models\\mc_dl_embedding_scaler.pkl']

### PyTorch Dataset i DataLoader

In [ ]:
class ProteinEmbeddingMultiLabelDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32

train_dataset = ProteinEmbeddingMultiLabelDataset(X_tr_scaled, y_tr)
val_dataset = ProteinEmbeddingMultiLabelDataset(X_val_scaled, y_val)
test_dataset = ProteinEmbeddingMultiLabelDataset(X_test_scaled, y_mc_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)